# DBSCAN Parameter Tuning & Distance Metric Comparison

This notebook tunes the from-scratch DBSCAN implementation (`models/dbscan.py`) by testing combinations of `eps` and `min_samples`, computes clustering metrics (ARI, AMI, silhouette score, homogeneity, completeness, purity), and compares the best parameter combination using both `euclidean` and `manhattan` distance metrics.

Plan:
1. Load the engineered dataset (scaled) for DBSCAN.
2. Run a grid search discretely over eps/min_samples and collect metrics for `euclidean` metric.
3. Visualize metric heatmaps and select the best combination (by ARI).
4. With the best parameters, compare `euclidean` vs `manhattan` metrics and visualize results.

In [ ]:
# Imports and path setup
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))
from config import FIRES_ENGINEERED

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (adjusted_rand_score, adjusted_mutual_info_score,
                             homogeneity_score, completeness_score, silhouette_score)
from sklearn.metrics import confusion_matrix

sns.set_style('whitegrid')
plt.style.use('seaborn-v0_8-darkgrid')

# Import our DBSCAN implementation
from models.dbscan import DBSCAN

print('✅ Imports loaded and custom DBSCAN available')

✅ Imports loaded and custom DBSCAN available


In [ ]:
# Load engineered dataset (scaled features)
df_eng = pd.read_csv(FIRES_ENGINEERED)
print('Engineered dataset shape:', df_eng.shape)
print(df_eng['class'].value_counts())

# Keep X (excluding lon/lat/class) and y for evaluation against true labels
X = df_eng.drop(['longitude', 'latitude', 'class'], axis=1).values
y_true = df_eng['class'].values

print('Feature matrix X shape:', X.shape, 'y shape:', y_true.shape)

Engineered dataset shape: (7302, 23)
class
0    4017
1    3285
Name: count, dtype: int64
Feature matrix X shape: (7302, 20) y shape: (7302,)


## Grid Search: EPS x MinSamples (euclidean)

We test a predefined grid of `eps` and `min_samples` and compute: number of clusters (excluding noise), number of noise points, ARI, AMI, Homogeneity, Completeness, Purity, and Silhouette Score (if applicable).

In [6]:
# Grid to test
# eps_values = [0.2, 0.3, 0.5, 0.8, 1.2]  
# min_samples_values = [3, 5, 10, 20]
eps_values = [3]  
min_samples_values = [20]
results = []

for eps in eps_values:
    for min_s in min_samples_values:
        model = DBSCAN(eps=eps, min_samples=min_s, metric='euclidean')
        labels = model.fit_predict(X)

        # Compute counts
        n_noise = np.sum(labels == -1)
        n_clusters = len(np.unique(labels[labels != -1]))

        # If fewer than 2 clusters (beyond noise), silhouette is not defined
        try:
            if n_clusters >= 2:
                sil = silhouette_score(X, labels, metric='euclidean')
            else:
                sil = np.nan
        except Exception:
            sil = np.nan

        # Supervised cluster quality metrics (compare to true labels)
        ari = adjusted_rand_score(y_true, labels)
        ami = adjusted_mutual_info_score(y_true, labels, average_method='arithmetic')
        hom = homogeneity_score(y_true, labels)
        comp = completeness_score(y_true, labels)

        # Purity: sum over clusters of majority class count / n_samples
        cm = confusion_matrix(y_true, labels, labels=np.unique(labels))
        # Contingency matrix mapping: rows are true classes, cols are clusters -> careful if label == -1 included
        # Build mapping cluster -> majority class counts (only cluster labels >= 0)
        purity = np.nan
        if n_clusters > 0:
            # compute contingency matrix with unique non-noise clusters
            # Build purity without requiring linear assignment: sum of max per cluster / n
            unique_clusters = np.unique(labels[labels != -1])
            cluster_purities = []
            for c in unique_clusters:
                members = y_true[labels == c]
                if len(members) == 0:
                    continue
                most_common = np.bincount(members.astype(int)).max()
                cluster_purities.append(most_common)
            purity = np.sum(cluster_purities) / len(labels)

        results.append({
            'eps': eps,
            'min_samples': min_s,
            'n_clusters': n_clusters,
            'n_noise': int(n_noise),
            'ARI': ari,
            'AMI': ami,
            'homogeneity': hom,
            'completeness': comp,
            'purity': purity,
            'silhouette': sil
        })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(['ARI','silhouette'], ascending=False)

print('Grid search complete. Top 5 combinations by ARI:')
display(results_df.head(10))

KeyboardInterrupt: 

## Visualize Grid Search Results

Create heatmaps (eps vs min_samples) for ARI and silhouette score (where defined) to locate the best combination.

In [ ]:
# Pivot results for heatmaps
def pivot_metric(df, metric):
    pivot = df.pivot(index='min_samples', columns='eps', values=metric)
    pivot = pivot.sort_index(ascending=True)
    return pivot

ari_pivot = pivot_metric(results_df, 'ARI')
sil_pivot = pivot_metric(results_df, 'silhouette')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(ari_pivot, annot=True, fmt='.3f', cmap='YlGnBu', ax=axes[0])
axes[0].set_title('Adjusted Rand Index (ARI)')
axes[0].set_xlabel('eps')
axes[0].set_ylabel('min_samples')

sns.heatmap(sil_pivot, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[1])
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('eps')
axes[1].set_ylabel('min_samples')

plt.tight_layout()
plt.show()

## Choose best parameters and compare metrics for Euclidean vs Manhattan

We'll pick the combination that maximizes ARI. If there are ties, we choose the combination with higher silhouette. Then rerun DBSCAN with this best pair for both distance metrics and compare metrics.

In [ ]:
# Get the best combination by ARI then silhouette
best_row = results_df.sort_values(['ARI','silhouette'], ascending=False).iloc[0]
best_eps = float(best_row['eps'])
best_min_samples = int(best_row['min_samples'])
print(f'Best combination (by ARI): eps={best_eps}, min_samples={best_min_samples}, ARI={best_row[
]:.4f}, Sil={best_row[
]}')

def evaluate_distance(metric):
    model = DBSCAN(eps=best_eps, min_samples=best_min_samples, metric=metric)
    labels = model.fit_predict(X)
    n_clusters = len(np.unique(labels[labels != -1]))
    n_noise = int(np.sum(labels == -1))
    ari = adjusted_rand_score(y_true, labels)
    ami = adjusted_mutual_info_score(y_true, labels, average_method='arithmetic')
    hom = homogeneity_score(y_true, labels)
    comp = completeness_score(y_true, labels)
    purity = np.nan
    if n_clusters > 0:
        cluster_purities = []
        unique_clusters = np.unique(labels[labels != -1])
        for c in unique_clusters:
            members = y_true[labels == c]
            if len(members) == 0:
                continue
            most_common = np.bincount(members.astype(int)).max()
            cluster_purities.append(most_common)
        purity = np.sum(cluster_purities) / len(labels)
    try:
        if n_clusters >= 2:
            sil = silhouette_score(X, labels, metric=metric)
        else:
            sil = np.nan
    except Exception:
        sil = np.nan
    return {
        'metric': metric,
        'n_clusters': n_clusters,
        'n_noise': n_noise,
        'ARI': ari,
        'AMI': ami,
        'homogeneity': hom,
        'completeness': comp,
        'purity': purity,
        'silhouette': sil
    }

metrics_euc = evaluate_distance('euclidean')
metrics_man = evaluate_distance('manhattan')

compare_df = pd.DataFrame([metrics_euc, metrics_man]).set_index('metric')
print('Comparison (best parameters):')
display(compare_df)


## Visual comparison of the distance metrics for the best parameters

Bar chart comparing ARI, AMI, Silhouette, Purity for Euclidean vs Manhattan

In [ ]:
metrics_to_plot = ['ARI','AMI','purity','silhouette']
vals = compare_df[metrics_to_plot]

fig, ax = plt.subplots(figsize=(10, 6))
vals.plot(kind='bar', ax=ax)
ax.set_title(f'Performance with best parameters (eps={best_eps}, min_samples={best_min_samples})')
ax.set_ylabel('Score (higher is better)')
ax.set_ylim(0, 1.05)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Show cluster counts for each metric
print('Cluster counts and noise')
display(compare_df[['n_clusters','n_noise']])

## Final Notes

- We used the engineered dataset (scaled features). If you wish, we can run the same grid on the original dataset (after numeric selection) and compare results.
- DBSCAN depends heavily on the scale of features; if you switch datasets or features, re-tune `eps`.
- We used ARI as the primary metric for selecting best parameters. If you prefer to prioritize Silhouette or Purity, we can select based on that instead.